In [ ]:
import os
import shutil
import pandas as pd
import ast
import random
from PIL import Image
from IPython.display import FileLink, display

!pip install ultralytics
from ultralytics import YOLO

## Set paths to all critical directories

In [ ]:
BASE_DEBRIS_DIR = '/kaggle/input/datasets/sadianawar/debris-detection-dataset/debris-detection'
EWA_EARTH_DIR = '/kaggle/input/datasets/ewakasprzak/earth-satellite-images-512x512/earth-satellite-images-dataset'
NEW_DATASET_DIR = '/kaggle/input/datasets/muhammadzakria2001/space-debris-detection-dataset-for-yolov8'

YOLO_DIR_EXP = "/kaggle/working/zorza_dataset"
YAML_PATH_EXP = "/kaggle/working/zorza.yaml"

## Helper functions for dataset preparation

In [ ]:
def copy_earth_data(src_base, split, dest_base):
    src_images_dir, src_labels_dir = os.path.join(src_base, split, 'images'), os.path.join(src_base, split, 'labels')
    dest_images_dir, dest_labels_dir = os.path.join(dest_base, split, 'images'), os.path.join(dest_base, split, 'labels')
    
    os.makedirs(dest_images_dir, exist_ok=True)
    os.makedirs(dest_labels_dir, exist_ok=True)
    if not os.path.exists(src_images_dir): return
        
    all_images = [f for f in os.listdir(src_images_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    for img_name in all_images:
        shutil.copy(os.path.join(src_images_dir, img_name), os.path.join(dest_images_dir, img_name))
        txt_name = os.path.splitext(img_name)[0] + '.txt'
        src_txt_path = os.path.join(src_labels_dir, txt_name)
        if os.path.exists(src_txt_path): shutil.copy(src_txt_path, os.path.join(dest_labels_dir, txt_name))
        else: open(os.path.join(dest_labels_dir, txt_name), 'w').close()

def process_debris_all(csv_file, src_img_dir, dest_img_dir, dest_label_dir):
    os.makedirs(dest_img_dir, exist_ok=True)
    os.makedirs(dest_label_dir, exist_ok=True)
    if not os.path.exists(csv_file): return

    df = pd.read_csv(csv_file)
    for idx, row in df.iterrows():
        image_id = row['ImageID']
        img_name = f"{image_id}.jpg"
        src_img_path = os.path.join(src_img_dir, img_name)
        
        if not os.path.exists(src_img_path): continue
        shutil.copy(src_img_path, os.path.join(dest_img_dir, img_name))
        with Image.open(src_img_path) as img: img_w, img_h = img.size
            
        try: boxes = ast.literal_eval(row['bboxes'])
        except: boxes = []
            
        with open(os.path.join(dest_label_dir, f"{image_id}.txt"), "w") as f_label:
            for box in boxes:
                xmin, xmax, ymin, ymax = map(float, box)
                w, h = xmax - xmin, ymax - ymin
                x_center, y_center = xmin + (w / 2.0), ymin + (h / 2.0)
                f_label.write(f"0 {x_center/img_w:.6f} {y_center/img_h:.6f} {w/img_w:.6f} {h/img_h:.6f}\n")

## Dataset preparation

In [ ]:
process_debris_all(os.path.join(BASE_DEBRIS_DIR, "train.csv"), os.path.join(BASE_DEBRIS_DIR, "train"), os.path.join(YOLO_DIR_EXP, "train", "images"), os.path.join(YOLO_DIR_EXP, "train", "labels"))
process_debris_all(os.path.join(BASE_DEBRIS_DIR, "val.csv"), os.path.join(BASE_DEBRIS_DIR, "val"), os.path.join(YOLO_DIR_EXP, "val", "images"), os.path.join(YOLO_DIR_EXP, "val", "labels"))

copy_earth_data(EWA_EARTH_DIR, 'train', YOLO_DIR_EXP)
copy_earth_data(EWA_EARTH_DIR, 'val', YOLO_DIR_EXP)

yaml_content = f"""
path: {YOLO_DIR_EXP}
train: train/images
val: val/images
names:
  0: space_debris
"""
with open(YAML_PATH_EXP, "w") as f: 
    f.write(yaml_content.strip())

train_img_cnt = len(os.listdir(os.path.join(YOLO_DIR_EXP, "train", "images")))
train_lbl_cnt = len(os.listdir(os.path.join(YOLO_DIR_EXP, "train", "labels")))
val_img_cnt = len(os.listdir(os.path.join(YOLO_DIR_EXP, "val", "images")))
val_lbl_cnt = len(os.listdir(os.path.join(YOLO_DIR_EXP, "val", "labels")))

print(f"Zbiór treningowy (train): {train_img_cnt} zdjęć | {train_lbl_cnt} etykiet")
print(f"Zbiór walidacyjny (val):  {val_img_cnt} zdjęć | {val_lbl_cnt} etykiet")
print(f"Suma: {train_img_cnt + val_img_cnt} zdjęć gotowych do treningu.")

## Model training
* for YOLOv8n set model = YOLO('yolov8n.pt')
* for YOLO26n set model = YOLO('yolo26n.pt')

In [ ]:
model = YOLO('yolov8n.pt')

results = model.train(
    data=YAML_PATH_EXP,  
    epochs=30,               
    imgsz=640,           
    batch=32,            
    device=0,            
    project='/kaggle/working/ZORZA',
    name='wersja_yolov8n', 
    save=True,
    patience=10          
)

metrics = model.val() 

print(f"🔹 mAP@50 (Dokładność ogólna detekcji): {metrics.box.map50:.4f}")
print(f"🔹 mAP@50-95 (Dokładność bounding boxów): {metrics.box.map:.4f}")

ZIP_NAME = '/kaggle/working/ZORZA_yolov8n'
shutil.make_archive(ZIP_NAME, 'zip', '/kaggle/working/ZORZA/yolov8n')

display(FileLink(os.path.basename(ZIP_NAME) + '.zip'))